# CITS4403 group project — committed sweep runner

This notebook is designed for **Save & Run All (commit)**. Everything it writes goes under `/kaggle/working/project/` (`data/`, `figures/`), which Kaggle preserves as version output. Cell printed tables are also kept in the committed notebook.

**Steps:** 1) locate code (dataset or local upload) → 2) deps → 3) verify `pilot_full.csv` (resume-safe) → 4) H2 low-budget zoom → 5) beta-sensitivity → 6) analysis + figures + summary file.


In [ ]:
# 1. locate project code (dataset copy preferred, fall back to any upload layout)
import glob, os, shutil, sys

PROJ_DATASET = '/kaggle/input/datasets/mike11088/4403project/4403PROJECT'
WORK = '/kaggle/working/project'

src = None
for cand in [PROJ_DATASET] + glob.glob('/kaggle/input/*/4403PROJECT') + glob.glob('/kaggle/input/*/*/*PROJECT*'):
    if cand and os.path.isdir(os.path.join(cand, 'src')):
        src = cand
        break
print('code source:', src)
assert src is not None, 'project code with src/ not found under /kaggle/input — attach the dataset'
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(src, WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print('copied to:', WORK)
print(sorted(os.listdir(WORK)))


In [ ]:
# 2. deps (kaggle image already ships these; best-effort for offline commits)
import importlib
for mod, pip in [('networkx', 'networkx'), ('pandas', 'pandas'), ('matplotlib', 'matplotlib'),
                 ('seaborn', 'seaborn'), ('scipy', 'scipy'), ('tqdm', 'tqdm')]:
    try:
        importlib.import_module(mod)
        print(mod, 'ok')
    except ImportError:
        print(mod, 'MISSING — run: !pip install -q', pip)


In [ ]:
# 3. verify main sweep (resume-safe: 0 todo if data/ came with the dataset)
from src.experiments import grid_configs, run_sweep
cfgs = grid_configs(topologies=('ws', 'ba', 'er'),
    rules=('observation', 'imitation', 'broadcast', 'broadcast_hub', 'none'),
    alphas=(0.0, 0.25, 0.5, 0.75, 1.0), deltas=(0.0, 0.05, 0.15, 0.3, 0.6),
    seeds=range(30), n=1000)
print('configs:', len(cfgs))
df = run_sweep(cfgs, f'{WORK}/data/pilot_full.csv', workers=4)
print(df.shape)
print(df.groupby(['topology', 'rule'])['final_size'].mean().round(3).to_string())


In [ ]:
# 4. H2 low-budget zoom: uniform vs hub-targeted at matched budgets (alpha<=0.15)
#    (~3600 runs, ~10-15 min on 4 workers; resume-safe on re-run)
from src.experiments import grid_configs, run_sweep
zcfgs = grid_configs(topologies=('ws', 'ba', 'er'),
    rules=('broadcast', 'broadcast_hub'),
    alphas=(0.02, 0.05, 0.08, 0.1, 0.15), deltas=(0.0, 0.05, 0.15, 0.3),
    seeds=range(30), n=1000)
print('configs:', len(zcfgs))
dfz = run_sweep(zcfgs, f'{WORK}/data/zoom_h2.csv', workers=4)
print(dfz.shape)
print(dfz.groupby(['topology', 'rule', 'alpha'])['final_size'].mean().round(3).to_string())


In [ ]:
# 5. beta-sensitivity: baseline saturation check (~600 runs)
from src.experiments import grid_configs, run_sweep
bcfgs = []
for b in [0.03, 0.06, 0.09, 0.12, 0.2]:
    bcfgs += grid_configs(topologies=('ws', 'ba', 'er'),
        rules=('none', 'observation'), alphas=(0.75,), deltas=(0.1,),
        seeds=range(20), n=1000, beta=b)
print('configs:', len(bcfgs))
dfb = run_sweep(bcfgs, f'{WORK}/data/beta_sweep.csv', workers=4)
print(dfb.shape)
print(dfb.groupby(['beta', 'topology', 'rule'])['final_size'].mean().round(3).to_string())


In [ ]:
# 6. analysis + figures — ALL written under /kaggle/working/project (kept on commit)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os

os.makedirs(f'{WORK}/figures', exist_ok=True)
df = pd.read_csv(f'{WORK}/data/pilot_full.csv')
dfz = pd.read_csv(f'{WORK}/data/zoom_h2.csv')
dfb = pd.read_csv(f'{WORK}/data/beta_sweep.csv')

# 6a. phase panels: observation final size + broadcast waves (hero figures)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, t in enumerate(['ws', 'ba', 'er']):
    sns.heatmap(df[(df.rule == 'observation') & (df.topology == t)]
        .pivot_table(index='delta', columns='alpha', values='final_size', aggfunc='mean'),
        ax=axes[0, i], cbar=(i == 2), vmin=0, vmax=1)
    axes[0, i].set_title(f'observation final_size on {t}')
    sns.heatmap(df[(df.rule == 'broadcast') & (df.topology == t)]
        .pivot_table(index='delta', columns='alpha', values='waves', aggfunc='mean'),
        ax=axes[1, i], cbar=(i == 2))
    axes[1, i].set_title(f'broadcast waves on {t}')
plt.tight_layout()
fig.savefig(f'{WORK}/figures/phase_panels.png', dpi=150)
plt.close(fig)

# 6b. efficiency table (RQ): reduction per unit social cost vs none-baseline
base = df[df.rule == 'none'].groupby('topology')['final_size'].mean()
agg = df[df.alpha > 0].groupby(['topology', 'rule'])[['final_size', 'social_cost_degree']].mean()
agg['reduction'] = [base[t] - f for (t, _), f in zip(agg.index, agg['final_size'])]
agg['eff'] = 1000 * agg['reduction'] / (agg['social_cost_degree'] + 1)
print(agg.to_string(float_format=lambda x: f'{x:.3e}'))

# 6c. H2 zoom figure: hub vs uniform final size vs budget, per topology
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for i, t in enumerate(['ws', 'ba', 'er']):
    sub = dfz[dfz.topology == t].groupby(['rule', 'alpha'])['final_size'].mean().reset_index()
    sns.lineplot(data=sub, x='alpha', y='final_size', hue='rule', marker='o', ax=axes[i])
    axes[i].set_title(f'H2 zoom on {t}')
    axes[i].set_xlabel('budget (alpha = p_comply)')
plt.tight_layout()
fig.savefig(f'{WORK}/figures/zoom_h2.png', dpi=150)
plt.close(fig)
zeff = dfz.groupby(['topology', 'rule'])[['final_size', 'social_cost_degree']].mean()
print(zeff.to_string(float_format=lambda x: f'{x:.3e}'))

# 6d. beta-sensitivity figure: baseline saturation
fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(data=dfb, x='beta', y='final_size', hue='rule', style='topology',
             markers=True, errorbar='sd', ax=ax)
ax.set_title('beta-sensitivity (observation alpha=0.75, delta=0.1)')
plt.tight_layout()
fig.savefig(f'{WORK}/figures/beta_sweep.png', dpi=150)
plt.close(fig)
print(dfb.groupby(['beta', 'topology', 'rule'])['final_size'].mean().round(3).to_string())

# 6e. commit summary file (also preserved in output)
with open(f'{WORK}/figures/commit_summary.txt', 'w') as f:
    f.write('CITS4403 committed run summary\n\n')
    f.write(f'pilot_full: {df.shape[0]} rows\n')
    f.write(f'zoom_h2: {dfz.shape[0]} rows\n')
    f.write(f'beta_sweep: {dfb.shape[0]} rows\n\n')
    f.write('efficiency (reduction per cost):\n' + agg.to_string() + '\n')
print('wrote figures/phase_panels.png figures/zoom_h2.png figures/beta_sweep.png figures/commit_summary.txt')


In [ ]:
# 7. verify committed outputs (everything here is preserved on Save & Run All)
!ls -R /kaggle/working/project/data /kaggle/working/project/figures
